# Bài 1: Bài Tập Thực Hành - Giới Thiệu LightGBM

**Hướng dẫn:**
- Hoàn thành các câu hỏi và bài tập dưới đây để củng cố kiến thức từ bài học.
- Cố gắng tự trả lời trước khi xem đáp án.
- Chạy các ô code để kiểm tra kết quả của bạn.

---

### Phần 1: Câu hỏi lý thuyết

**Câu 1:** Trình bày ý tưởng cốt lõi của thuật toán Gradient Boosting. Theo bạn, điều gì làm cho nó khác biệt so với các thuật toán ensemble khác như Random Forest?

**Câu 2:** LightGBM sử dụng chiến lược phát triển cây là "Leaf-wise". Hãy giải thích chiến lược này và so sánh nó với chiến lược "Level-wise" mà XGBoost thường dùng. Ưu và nhược điểm của "Leaf-wise" là gì?

**Câu 3:** Kỹ thuật Gradient-based One-Side Sampling (GOSS) giúp LightGBM tăng tốc độ huấn luyện như thế nào? Nó dựa trên giả định nào về dữ liệu?

**Câu 4:** Exclusive Feature Bundling (EFB) là gì và nó giải quyết vấn đề gì trong các tập dữ liệu lớn?

**Câu 5:** Tại sao việc xử lý các biến categorical (phân loại) lại quan trọng đối với các mô hình cây quyết định? LightGBM cung cấp cơ chế nào để xử lý chúng một cách hiệu quả?

---

### Phần 2: Bài tập thực hành

**Bối cảnh:** Chúng ta sẽ sử dụng bộ dữ liệu về dự đoán bệnh tim (`heart.csv`). Mục tiêu là xây dựng một mô hình LightGBM để dự đoán xem một bệnh nhân có bị bệnh tim hay không (`target`).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import accuracy_score

# Tải dữ liệu
url = 'https://raw.githubusercontent.com/justmarkham/DAT8/master/data/heart.csv'
df_heart = pd.read_csv(url)

df_heart.head()

**Bài tập 6: Chuẩn bị dữ liệu**

1.  Tách biến mục tiêu (`target`) và các biến đặc trưng (features) ra thành `X` và `y`.
2.  Phân chia dữ liệu thành tập huấn luyện (`X_train`, `y_train`) và tập kiểm tra (`X_test`, `y_test`) với tỷ lệ 80/20. Sử dụng `random_state=10` để đảm bảo kết quả có thể tái lập.

In [ ]:
# Viết code của bạn vào đây
X = ...
y = ...

X_train, X_test, y_train, y_test = ...

**Bài tập 7: Huấn luyện mô hình LightGBM**

1.  Khởi tạo một mô hình `lgb.LGBMClassifier`.
2.  Đặt `random_state=42` để kết quả nhất quán.
3.  Huấn luyện mô hình trên tập `train`.

In [ ]:
# Viết code của bạn vào đây
lgbm = ...

lgbm.fit(...)

**Bài tập 8: Đánh giá mô hình**

1.  Dùng mô hình đã huấn luyện để dự đoán nhãn trên tập `test`.
2.  Tính và in ra độ chính xác (accuracy) của mô hình.

In [ ]:
# Viết code của bạn vào đây
y_pred = ...
accuracy = ...

print(f"Độ chính xác của mô hình trên tập test là: {accuracy:.4f}")

---

## Đáp Án

### Phần 1: Lý thuyết

**Câu 1:**
Ý tưởng cốt lõi của Gradient Boosting là xây dựng một chuỗi các mô hình yếu (thường là cây quyết định), trong đó mỗi mô hình sau được huấn luyện để sửa chữa sai số (residual) của mô hình trước đó. Nó hoạt động tuần tự: mô hình 2 học từ lỗi của mô hình 1, mô hình 3 học từ lỗi còn lại của (mô hình 1 + mô hình 2), v.v.
**Sự khác biệt với Random Forest:**
- **Tuần tự vs. Song song:** Gradient Boosting xây dựng các cây một cách tuần tự. Random Forest xây dựng các cây một cách độc lập và song song.
- **Học từ lỗi:** Cây trong Gradient Boosting học từ sai lầm của các cây trước. Cây trong Random Forest không liên quan đến nhau.
- **Trọng số:** Gradient Boosting kết hợp các cây yếu bằng cách cộng dồn có trọng số. Random Forest thường kết hợp bằng cách lấy trung bình (cho regression) hoặc bỏ phiếu đa số (cho classification).

**Câu 2:**
- **Level-wise (Bredth-first):** Phát triển cây theo từng tầng. Tất cả các lá ở cùng một độ sâu sẽ được xem xét để phân chia. Điều này tạo ra cây cân bằng nhưng có thể lãng phí tài nguyên vào việc phân chia những nhánh không mang lại nhiều lợi ích.
- **Leaf-wise (Depth-first):** Không phát triển theo tầng, mà ở mỗi bước, nó sẽ tìm và phân chia nhánh lá nào mang lại mức giảm loss lớn nhất. Điều này giúp hội tụ nhanh hơn và tạo ra mô hình chính xác hơn.
  - **Ưu điểm:** Tốc độ hội tụ nhanh, độ chính xác thường cao hơn.
  - **Nhược điểm:** Dễ bị overfitting trên các tập dữ liệu nhỏ, vì nó có xu hướng tạo ra các cây rất sâu và phức tạp. Cần kiểm soát bằng các tham số như `num_leaves` hoặc `max_depth`.

**Câu 3:**
GOSS (Gradient-based One-Side Sampling) tăng tốc độ bằng cách giảm số lượng mẫu dữ liệu được sử dụng để tính toán information gain. Nó hoạt động như sau:
1.  Giữ lại tất cả các mẫu có gradient lớn (đây là những mẫu mà mô hình hiện tại dự đoán rất sai).
2.  Lấy một mẫu ngẫu nhiên từ các mẫu có gradient nhỏ.
Giả định đằng sau là các mẫu bị dự đoán sai nhiều (gradient lớn) chứa nhiều thông tin hữu ích hơn cho việc cải thiện mô hình. Bằng cách tập trung vào chúng và chỉ lấy một phần nhỏ các mẫu "dễ đoán", GOSS giảm đáng kể khối lượng tính toán mà vẫn giữ được độ chính xác.

**Câu 4:**
EFB (Exclusive Feature Bundling) là một kỹ thuật để giảm số lượng đặc trưng (features) trong các tập dữ liệu có chiều cao và thưa (sparse). Nó giải quyết vấn đề chi phí tính toán cao khi tìm điểm chia tối ưu trên hàng ngàn đặc trưng.
EFB hoạt động bằng cách gộp các đặc trưng mà gần như không bao giờ có giá trị khác 0 cùng một lúc (mutually exclusive) thành một đặc trưng duy nhất (bundle). Ví dụ, các cột được tạo từ one-hot encoding là các ứng cử viên hoàn hảo. Việc này giảm số lượng đặc trưng một cách an toàn, giúp thuật toán tìm điểm chia nhanh hơn rất nhiều.

**Câu 5:**
Các mô hình cây quyết định về bản chất hoạt động bằng cách chia các biến số (numerical) tại một ngưỡng (ví dụ: `tuổi > 30`). Chúng không thể xử lý trực tiếp các biến dạng text hoặc categorical (ví dụ: `thành phố = 'Hà Nội'`). Do đó, các biến này cần được chuyển đổi sang dạng số.
LightGBM có cơ chế xử lý biến categorical rất hiệu quả. Thay vì phải thực hiện one-hot encoding (có thể tạo ra quá nhiều cột), bạn có thể chỉ định các cột là `category`. LightGBM sẽ sử dụng một thuật toán phân chia tối ưu (Fisher's method) để tìm cách nhóm các category lại với nhau một cách hiệu quả, giúp giảm độ phức tạp và tăng tốc độ.

### Phần 2: Thực hành (Code đáp án)

In [ ]:
# Bài tập 6: Chuẩn bị dữ liệu
X = df_heart.drop('target', axis=1)
y = df_heart['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

print(f"Kích thước tập train: {X_train.shape}")
print(f"Kích thước tập test: {X_test.shape}")

In [ ]:
# Bài tập 7: Huấn luyện mô hình LightGBM
lgbm = lgb.LGBMClassifier(random_state=42)

lgbm.fit(X_train, y_train)

print("Mô hình đã được huấn luyện thành công.")

In [ ]:
# Bài tập 8: Đánh giá mô hình
y_pred = lgbm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Độ chính xác của mô hình trên tập test là: {accuracy:.4f}")